# SNF Feature Engineering And Threshold Baselines

**Executive takeaway:** The automated flagger uses leakage-safe, facility-normalized SNF features so administrators do not have to rely only on manually configured gross pay, total hours, overtime, or premium-dollar thresholds.

In [1]:
from common.plots import LetsPlot, aes, geom_point, ggplot, labs, theme_minimal

from payroll_anomaly_ranking.columns import FeatureCol, PayrollCol, RuleCol, ScoreCol
from payroll_anomaly_ranking.config import PayrollConfig
from payroll_anomaly_ranking.features import build_features
from payroll_anomaly_ranking.pipeline import run_pipeline
from payroll_anomaly_ranking.rules import add_rule_flags

LetsPlot.setup_html()

config = PayrollConfig(employee_count=160, pay_periods=12, review_budgets=(10, 25))
results = run_pipeline(config)
featured = add_rule_flags(build_features(results.payroll))

## Leakage-Safe SNF Features

Historical features use prior shifts and prior pay periods. Peer features normalize within facility, role, shift type, unit, and pay-code context. Synthetic labels remain evaluation-only and are not used as model features, threshold baselines, exposure inputs, or administrator queue fields.

In [2]:
featured.select(
    [
        PayrollCol.EMPLOYEE_ID,
        PayrollCol.FACILITY_ID,
        PayrollCol.ROLE,
        PayrollCol.SHIFT_DATE,
        PayrollCol.SHIFT_TYPE,
        PayrollCol.SCHEDULED_HOURS,
        PayrollCol.PAID_HOURS,
        PayrollCol.OVERTIME_HOURS,
        PayrollCol.PREMIUM_PAY,
        FeatureCol.OVERTIME_PER_SCHEDULED_HOUR,
        FeatureCol.PAID_MINUS_SCHEDULED_HOURS,
        FeatureCol.PREMIUM_PAY_SHARE,
        FeatureCol.GROSS_TO_EXPECTED_SHIFT_PAY,
        FeatureCol.PREMIUM_ELIGIBILITY_MISMATCH,
        FeatureCol.REST_GAP_RISK,
        RuleCol.REASON_CODES,
    ],
).head(12)

employee_id,facility_id,role,shift_date,shift_type,scheduled_hours,paid_hours,overtime_hours,premium_pay,overtime_per_scheduled_hour,paid_minus_scheduled_hours,premium_pay_share,gross_to_expected_shift_pay,premium_eligibility_mismatch,rest_gap_risk,rule_reason_codes
str,str,str,date,str,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,str
"""SYN-SNF-E00001""","""SNF-F005""","""CNA""",2024-01-05,"""Evening""",8.0,7.86,0.0,17.69,0.0,-0.14,0.102444,0.984268,0,0,"""none"""
"""SYN-SNF-E00001""","""SNF-F005""","""CNA""",2024-01-10,"""Day""",8.0,7.92,0.0,0.0,0.0,-0.08,0.0,0.989985,0,0,"""none"""
"""SYN-SNF-E00001""","""SNF-F005""","""CNA""",2024-01-21,"""Evening""",8.0,7.83,0.0,33.28,0.0,-0.17,0.177314,0.982464,0,0,"""none"""
"""SYN-SNF-E00001""","""SNF-F005""","""CNA""",2024-01-25,"""Night""",8.0,8.08,0.08,28.28,0.01,0.08,0.150098,1.012739,0,0,"""none"""
"""SYN-SNF-E00001""","""SNF-F005""","""CNA""",2024-01-30,"""Evening""",8.0,7.81,0.0,17.57,0.0,-0.19,0.102395,0.978669,0,0,"""none"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SYN-SNF-E00001""","""SNF-F005""","""CNA""",2024-02-02,"""Evening""",8.0,7.55,0.0,16.99,0.0,-0.45,0.10243,0.949185,0,0,"""none"""
"""SYN-SNF-E00001""","""SNF-F005""","""CNA""",2024-02-04,"""Night""",8.0,8.0,0.0,44.0,0.0,0.0,0.218081,1.0,0,0,"""none"""
"""SYN-SNF-E00001""","""SNF-F005""","""CNA""",2024-02-07,"""Day""",8.0,8.12,0.12,0.0,0.015,0.12,0.0,1.022503,0,0,"""none"""


## Manual Threshold Baselines

These threshold flags approximate what many production workflows do today: configure cutoffs on individual fields. The automated model keeps those baselines for comparison but adds SNF context.

In [3]:
results.scored.select(
    [
        PayrollCol.FACILITY_ID,
        PayrollCol.ROLE,
        PayrollCol.SHIFT_TYPE,
        PayrollCol.GROSS_PAY,
        PayrollCol.PAID_HOURS,
        PayrollCol.OVERTIME_HOURS,
        PayrollCol.PREMIUM_PAY,
        ScoreCol.THRESHOLD_GROSS_PAY_FLAG,
        ScoreCol.THRESHOLD_TOTAL_HOURS_FLAG,
        ScoreCol.THRESHOLD_OVERTIME_HOURS_FLAG,
        ScoreCol.THRESHOLD_PREMIUM_DOLLARS_FLAG,
        ScoreCol.THRESHOLD_PAID_VS_SCHEDULED_FLAG,
        ScoreCol.FINAL_APPROVAL_EXCEPTION_SCORE,
    ],
).sort(ScoreCol.FINAL_APPROVAL_EXCEPTION_SCORE, descending=True).head(12)

facility_id,role,shift_type,gross_pay,paid_hours,overtime_hours,premium_pay,threshold_gross_pay_flag,threshold_total_hours_flag,threshold_overtime_hours_flag,threshold_premium_dollars_flag,threshold_paid_vs_scheduled_flag,final_approval_exception_score
str,str,str,f64,f64,f64,f64,i64,i64,i64,i64,i64,f64
"""SNF-F006""","""RN""","""Double""",1073.57,17.88,9.88,35.49,0,1,1,0,0,0.998085
"""SNF-F004""","""Dietary""","""Double""",414.54,17.55,9.55,61.58,0,1,1,0,0,0.996974
"""SNF-F002""","""RN""","""Double""",1112.16,17.73,9.73,16.08,0,1,1,0,0,0.989197
"""SNF-F002""","""Admin""","""Double""",866.78,17.92,9.92,34.63,0,1,1,0,0,0.987993
"""SNF-F006""","""CNA""","""Double""",367.78,17.55,9.55,28.66,0,1,1,0,0,0.987409
…,…,…,…,…,…,…,…,…,…,…,…,…
"""SNF-F004""","""Therapy""","""Double""",1022.49,17.72,9.72,16.1,0,1,1,0,0,0.981561
"""SNF-F002""","""RN""","""Double""",1088.08,17.62,9.62,0.0,0,1,1,0,0,0.97754
"""SNF-F005""","""RN""","""Double""",917.23,17.81,9.81,0.0,0,1,1,0,0,0.972992


In [4]:
plot_data = results.scored.select(
    PayrollCol.OVERTIME_HOURS,
    PayrollCol.PREMIUM_PAY,
    ScoreCol.FINAL_APPROVAL_EXCEPTION_SCORE,
    PayrollCol.IS_ANOMALY,
)
(
    ggplot(plot_data, aes(PayrollCol.OVERTIME_HOURS, PayrollCol.PREMIUM_PAY))
    + geom_point(aes(color=ScoreCol.FINAL_APPROVAL_EXCEPTION_SCORE), alpha=0.55)
    + labs(
        title="Automated approval score combines overtime and premium context",
        x="Overtime hours",
        y="Premium dollars",
        color="Approval score",
    )
    + theme_minimal()
)

CheckedPlot(_plot=<lets_plot.plot.core.PlotSpec object at 0x7f5b874a3240>)

## What This Proves

Stationary ratios, premium eligibility checks, rest-gap context, and facility-normalized peer comparisons provide more administrator-relevant signal than one-field threshold rules.